In [ ]:
import re
import pandas as pd
from snorkel.labeling import labeling_function, PandasLFApplier
from snorkel.labeling.model import LabelModel, MajorityLabelVoter
import torch
import numpy as np
from snorkel.labeling.analysis import LFAnalysis
import textwrap

# Snorkel 
Snorkel is used for the component-based reconstruction of the ChadsVasc score. In this notebook, all the logic related to Snorkel is grouped. 

The notebook starts by defining the function templates, followed by the definitions of the labelling functions. 

In [ ]:
FILE = r"..\..\Preprocessing\datasets\train_set.feather"
df = pd.read_feather(FILE)

print(len(df))
print(df.columns)

4618
Index(['diagnose_i480', 'diagnose_i481', 'diagnose_i482', 'diagnose_i483',
       'diagnose_i484', 'diagnose_i489', 'diagnose_i501', 'diagnose_i502',
       'diagnose_i503', 'diagnose_i504', 'diagnose_i508', 'diagnose_i509',
       'diagnose_i69', 'diagnose_n171819', 'diagnose_z8673', 'diagnose_i10',
       'diagnose_i11', 'diagnose_i12', 'diagnose_i13', 'diagnose_i15',
       'diagnose_e10', 'diagnose_e11', 'diagnose_e12', 'diagnose_e13',
       'diagnose_e14', 'uppercase_titles', 'num_titles', 'sections',
       'text_length', 'expert_chadsvasc_score', 'text', 'unique_id'],
      dtype='object')


In [ ]:
i48_cols = [col for col in df.columns if col.startswith("diagnose_i48")]
df_i48 = df[df[i48_cols].any(axis=1)]

print(f"Remaining rows after requiring AF diagnoses: {len(df_i48)}")
df = df_i48.copy()

Remaining rows after requiring AF diagnoses: 4618


In [ ]:
df = df.dropna(subset=["text"])
df = df.reset_index(drop=True)

print("Remaining rows:", len(df))

Remaining rows: 4618


# Snorkel: function templates

In [ ]:
ABSTAIN = -1
ABSENT = 0
PRESENT = 1

lfs_for_component = {}

FAM_KEYWORDS = [
    "familie",
    "familiale",
    "erfelijk",
    "erfelijkheid",
    "moeder",
    "vader",
    "broer",
    "zus",
]

## Using unstructured information, note: medication is treated as a separate component ##
relevant_sections = [
    "BESLUIT",
    "ANTECEDENTEN",
    "RISICOFACTOREN",
    "ANAMNESE",
]

In [ ]:
# def make_positive_lf(pos_list):
#     """
#     Factory function to create a positive LF for any list of positive concepts.

#     Parameters:
#         pos_list (list): List of regex patterns for the positive concepts.
#         neg_keywords (list): List of negation keywords.
#         window (int): Number of words around the concept to look for negation.

#     Returns:
#         labeling_function: A Snorkel LF that returns PRESENT if any positive concept is found.
#     """
#     @labeling_function(name="positive_lf")
#     def lf(x):
#         text = ' '.join(str(x['sections'].get(sec, '')).lower() for sec in relevant_sections)
#         text = x['text'].lower() if pd.notnull(x['text']) else ''
#         if any(re.search(p, text) for p in pos_list):
#             return PRESENT
#         return ABSTAIN

#     return lf


# Negation-aware positive LF
def make_positive_lf(
    pos_list,
    pre_keywords=["geen", "niet", "zonder"],
    post_keywords=["negatief", "afwezig"],
):
    @labeling_function(name="positive_lf")
    def lf(x):
        text = str(x.get("text", "")).lower()
        if not text:
            return ABSTAIN

        lines = text.splitlines()
        for line in lines:
            for pattern in pos_list:
                match = re.search(pattern, line)
                if match:
                    start, end = match.span()
                    prefix = line[:start]
                    suffix = line[end:]

                    # If it's negated on this line, this LF should ABSTAIN
                    # (letting the negation_lf handle it)
                    if any(neg in prefix for neg in pre_keywords):
                        continue
                    if any(neg in suffix for neg in post_keywords):
                        continue

                    # If we found the match and NO negation keywords on this line
                    return PRESENT

        return ABSTAIN

    return lf


def make_negation_lf(
    pos_list,
    pre_keywords=["geen", "niet", "zonder"],
    post_keywords=["negatief", "afwezig", "uitgesloten"],
):
    @labeling_function(name="negation_lf")
    def lf(x):
        text = str(x.get("text", "")).lower()
        if not text:
            return ABSTAIN

        lines = text.splitlines()

        for line in lines:
            for concept_pattern in pos_list:
                match = re.search(concept_pattern, line)
                if match:
                    start_char, end_char = match.span()

                    # 1. Check BEFORE the concept (Prefix)
                    prefix_text = line[:start_char]
                    if any(neg in prefix_text for neg in pre_keywords):
                        return ABSENT

                    # 2. Check AFTER the concept (Suffix)
                    suffix_text = line[end_char:]
                    if any(neg in suffix_text for neg in post_keywords):
                        return ABSENT

        return ABSTAIN

    return lf


def make_family_lf(pos_list):

    @labeling_function(name="family_lf")
    def lf(x):
        text = x["text"].lower() if pd.notnull(x["text"]) else ""
        lines = text.split("\n")

        for line in lines:
            for concept_pattern in pos_list:
                for match in re.finditer(concept_pattern, line):

                    if any(fam in line for fam in FAM_KEYWORDS):
                        return ABSENT

        return ABSTAIN

    return lf

# Labelling functions

## Labelling functions for CHF

In [ ]:
## Using structured information ##
@labeling_function()
def lf_structured_chf(x):
    chf_cols = [
        "diagnose_i501",
        "diagnose_i502",
        "diagnose_i503",
        "diagnose_i504",
        "diagnose_i508",
        "diagnose_i509",
    ]
    if x[chf_cols].any():
        return PRESENT
    else:
        return ABSTAIN


CHF_POS = [r"\bhartfalen\b", r"\bcordecompensatie\b", r"\bhartdecompensatie\b"]


lf_chf_positive = make_positive_lf(CHF_POS)
lf_chf_negation = make_negation_lf(CHF_POS)

# pattern to catch table-style mentions like "Congestief hartfalen - 0/1"
CHF_TABLE_PATTERN_1 = re.compile(
    r"\b(congestief hartfalen|hartfalen|CHF)\b\s*-\s:*(\d+)/\d+", re.IGNORECASE
)

CHF_TABLE_PATTERN_2 = re.compile(
    r"\b(congestief hartfalen|hartfalen|CHF)\b\s*:\s*([+-])", re.IGNORECASE
)


@labeling_function()
def lf_chf_tabular_format(x):
    """
    Detect CHF in tabular / structured lines like "Congestief hartfalen - 0/1"
    Returns:
        PRESENT if value == 1
        ABSENT if value == 0
        ABSTAIN otherwise
    """
    text = " ".join(
        str(x["sections"].get(sec, "")).lower() for sec in relevant_sections
    )
    text = x["text"].lower() if pd.notnull(x["text"]) else ""

    match = CHF_TABLE_PATTERN_1.search(text)
    if match:
        value = int(match.group(2))
        if value == 1:
            return PRESENT
        elif value == 0:
            return ABSENT
    return ABSTAIN


@labeling_function()
def lf_chf_tabular_format_2(x):
    text = " ".join(
        str(x["sections"].get(sec, "")).lower() for sec in relevant_sections
    )
    text = x["text"].lower() if pd.notnull(x["text"]) else ""

    match = CHF_TABLE_PATTERN_2.search(text)
    if match:
        value = 1 if match.group(2) == "+" else 0
        return PRESENT if value == 1 else ABSENT
    return ABSTAIN


CHF_RULES = [
    lambda text: re.search(r"\bjardiance\b", text) or re.search(r"\baldactone\b", text),
    lambda text: re.search(r"\bforxiga\b", text) or re.search(r"\baldactone\b", text),
    lambda text: re.search(r"\bburinex\b", text) or re.search(r"\blasix\b", text),
]


@labeling_function()
def lf_chf_medication(x):
    relevant_sections_medication = ["MEDICATIE", "ONTSLAG_MEDICATIE"]

    text = " ".join(
        str(x["sections"].get(sec, "")).lower() for sec in relevant_sections_medication
    )

    if any(rule(text) for rule in CHF_RULES):
        return PRESENT
    return ABSTAIN


lf_chf_family = make_family_lf(CHF_POS)


lfs_for_component = {
    "C": [
        lf_structured_chf,
        lf_chf_positive,
        lf_chf_negation,
        lf_chf_tabular_format,
        lf_chf_tabular_format_2,
        lf_chf_medication,
        lf_chf_family,
    ],
}

In [ ]:
comp = "C"
lfs = lfs_for_component[comp]
applier = PandasLFApplier(lfs=lfs)
L = applier.apply(df)
LFAnalysis(L, lfs_for_component["C"]).lf_summary()

100%|██████████| 4618/4618 [00:30<00:00, 153.81it/s]


,j,Polarity,Coverage,Overlaps,Conflicts
lf_structured_chf,0,[1],0.550022,0.530533,0.102858
positive_lf,1,[1],0.526418,0.517757,0.127977
negation_lf,2,[0],0.132741,0.127111,0.127111
lf_chf_tabular_format,3,"[0, 1]",0.003032,0.003032,0.001732
lf_chf_tabular_format_2,4,[0],0.002815,0.002815,0.002815
lf_chf_medication,5,[1],0.621481,0.511477,0.108922
family_lf,6,[0],0.008012,0.008012,0.008012


In [ ]:
df_copy = df.copy()
df_copy["lf_chf_negation"] = df_copy.apply(lf_chf_negation, axis=1)

chf_negation = df_copy[df_copy["lf_chf_negation"] == ABSENT]

print(f"Family LF triggered in {len(chf_negation)} rows.\n")

for i, row in chf_negation.head(15).iterrows():
    text = " ".join(str(row["sections"].get(sec, "")) for sec in relevant_sections)
    print(f"--- Row {i} ---")
    print(text)
    print("-------------------\n")

In [ ]:
# 1. Define the list of LFs you want to check against
chf_lfs = [
    lf_structured_chf,
    lf_chf_positive,
    lf_chf_tabular_format,
    lf_chf_tabular_format_2,
    lf_chf_medication,
    lf_chf_family,
]

# 2. Filter for rows where negation triggered
df_copy = df.copy()
df_copy["negation_result"] = df_copy.apply(lf_chf_negation, axis=1)
chf_negation_rows = df_copy[df_copy["negation_result"] == ABSENT]

print(f"Negation LF triggered in {len(chf_negation_rows)} rows.\n")

# 3. Iterate and check for other triggers
for i, row in chf_negation_rows.head(15).iterrows():
    triggered_others = []

    for lf in chf_lfs:
        if lf(row) == PRESENT:
            triggered_others.append(lf.name)

    text = " ".join(str(row["sections"].get(sec, "")) for sec in relevant_sections)

    print(f"--- Row {i} ---")
    print(
        f"CONFLICTS WITH: {', '.join(triggered_others) if triggered_others else 'None'}"
    )
    print(f"TEXT: {text[:1500]}...")  # Truncated for readability
    print("-------------------\n")

## Labelling functions for prior stroke

In [ ]:
@labeling_function()
def lf_structured_stroke(x):
    # Check the ICD columns for stroke
    stroke_cols = ["diagnose_i69", "diagnose_z8673"]
    if x[stroke_cols].any():
        return PRESENT
    else:
        return ABSTAIN


STROKE_POS = [r"\btia\b", r"\bcva\b"]

lf_stroke_positive = make_positive_lf(STROKE_POS)
lf_stroke_negation = make_negation_lf(STROKE_POS)
lf_stroke_family = make_family_lf(STROKE_POS)

lfs_for_component["S"] = [
    lf_structured_stroke,
    lf_stroke_positive,
    lf_stroke_negation,
    lf_stroke_family,
]

In [ ]:
comp = "S"
lfs = lfs_for_component[comp]
applier = PandasLFApplier(lfs=lfs)
L = applier.apply(df)
LFAnalysis(L, lfs_for_component["S"]).lf_summary()

100%|██████████| 4618/4618 [00:21<00:00, 219.25it/s]


,j,Polarity,Coverage,Overlaps,Conflicts
lf_structured_stroke,0,[1],0.129926,0.113469,0.008229
positive_lf,1,[1],0.142053,0.118450,0.013209
negation_lf,2,[0],0.009528,0.007796,0.007579
family_lf,3,[0],0.008878,0.008878,0.008662


## Labeling functions for hypertension
We use the sections again to avoid the family history when extracting mentions of hypertension.

We definitly need more labeling function. An idea would be to get a list of relevant hypertension medications (plus medications that are definitly not used in case of hypertension), and construct additional labeling functions around those facts. 

In [ ]:
@labeling_function()
def lf_structured_htn(x):
    cols = [
        "diagnose_i10",
        "diagnose_i11",
        "diagnose_i12",
        "diagnose_i13",
        "diagnose_i15",
    ]
    if x[cols].any():
        return PRESENT
    else:
        return ABSTAIN


HYPERTENSION_POS = [r"\barteriële hypertensie\b"]

lf_htn_positive = make_positive_lf(HYPERTENSION_POS)
lf_htn_negation = make_negation_lf(HYPERTENSION_POS)
lf_htn_family = make_family_lf(HYPERTENSION_POS)

HTN_TABLE_PATTERN = re.compile(r"\b(hypertensie)\b\s*:\s*([+-])", re.IGNORECASE)


@labeling_function()
def lf_htn_tabular_format(x):
    text = " ".join(
        str(x["sections"].get(sec, "")).lower() for sec in relevant_sections
    )
    text = x["text"].lower() if pd.notnull(x["text"]) else ""

    match = HTN_TABLE_PATTERN.search(text)
    if match:
        value = 1 if match.group(2) == "+" else 0
        return PRESENT if value == 1 else ABSENT
    return ABSTAIN


lfs_for_component["H"] = [
    lf_structured_htn,
    lf_htn_negation,
    lf_htn_positive,
    lf_htn_tabular_format,
    lf_htn_family,
]

In [ ]:
# comp = 'H'
# lfs = lfs_for_component[comp]
# applier = PandasLFApplier(lfs=lfs)
# L = applier.apply(df)
# LFAnalysis(L, lfs_for_component["H"]).lf_summary()

In [ ]:
# df_copy = df.copy()
# df_copy['lf_htn_negation'] = df_copy.apply(lf_htn_negation, axis=1)

# negation_triggered = df_copy[df_copy['lf_htn_negation'] == ABSENT]

# print(f"Negation LF triggered in {len(negation_triggered)} rows.\n")

# for i, row in negation_triggered.head(10).iterrows():
#     text = ' '.join(str(row['sections'].get(sec, '')) for sec in relevant_sections)
#     print(f"--- Row {i} ---")
#     print(text)
#     print("-------------------\n")

## Labeling functions for diabetes

In [ ]:
@labeling_function()
def lf_structured_dm(x):
    cols = [
        "diagnose_e10",
        "diagnose_e11",
        "diagnose_e12",
        "diagnose_e13",
        "diagnose_e14",
    ]
    if x[cols].any():
        return PRESENT
    else:
        return ABSTAIN


DIABETES_POS = [
    r"\bdiabetes mellitus\b",
    r"\bdiabetes\b",
    r"\bDM",
    # r'\bglucosemetabolisme\b',
]

DIABETES_MEDICATIONS = [r"\bmetformine\b", r"\bmetformax\b"]

lf_dm_positive = make_positive_lf(DIABETES_POS)
lf_dm_negation = make_negation_lf(DIABETES_POS)
lf_dm_family = make_family_lf(DIABETES_POS)


@labeling_function()
def lf_dm_medication(x):
    relevant_sections_medication = ["MEDICATIE", "ONTSLAG_MEDICATIE"]

    text = " ".join(
        str(x["sections"].get(sec, "")).lower() for sec in relevant_sections_medication
    )

    if any(re.search(p, text) for p in DIABETES_MEDICATIONS):
        return PRESENT
    return ABSTAIN


DM_TABLE_PATTERN = re.compile(
    r"\b(diabetes|glucosemetabolisme)\b\s*:\s*([+-])", re.IGNORECASE
)


@labeling_function()
def lf_dm_tabular_format(x):
    text = " ".join(
        str(x["sections"].get(sec, "")).lower() for sec in relevant_sections
    )
    text = x["text"].lower() if pd.notnull(x["text"]) else ""

    match = DM_TABLE_PATTERN.search(text)
    if match:
        value = 1 if match.group(2) == "+" else 0
        return PRESENT if value == 1 else ABSENT
    return ABSTAIN


lfs_for_component["D"] = [
    lf_structured_dm,
    lf_dm_negation,
    lf_dm_positive,
    lf_dm_medication,
    lf_dm_tabular_format,
    lf_dm_family,
]

In [ ]:
# comp = 'D'
# lfs = lfs_for_component[comp]
# applier = PandasLFApplier(lfs=lfs)
# L = applier.apply(df)
# LFAnalysis(L, lfs_for_component["D"]).lf_summary()

In [ ]:
# df_copy = df.copy()
# df_copy['lf_dm_positive'] = df_copy.apply(lf_dm_positive, axis=1)

# family_triggered = df_copy[df_copy['lf_dm_positive'] == PRESENT]

# print(f"Family LF triggered in {len(family_triggered)} rows.\n")

# for i, row in family_triggered.head(15).iterrows():
#     text = ' '.join(str(row['sections'].get(sec, '')) for sec in relevant_sections)
#     print(f"--- Row {i} ---")
#     print(text)
#     print("-------------------\n")

## Labelling functions for vascular disease

In [ ]:
VASCULAR_DISEASE_POS = [
    r"\bcoronair lijden\b",
    r"\bperifeer vaatlijden\b",
    r"\barterieel vaatlijden\b",
    r"takslijden\b",
    r"taksziekte\b",
    r"pta\b",
    r"ptca\b",
    r"pci\b",
    r"cabg\b",
    r"vasculopathie",
    r"coronaire bypass",
]


VASCULAR_MEDICATIONS = [
    r"\bclopidogrel\b",
    r"\bxarelto 2.5 mg\b",
    r"\bbrilique\b",
]

lf_vasc_positive = make_positive_lf(VASCULAR_DISEASE_POS)
lf_vasc_negation = make_negation_lf(VASCULAR_DISEASE_POS)
lf_vasc_family = make_family_lf(VASCULAR_DISEASE_POS)


@labeling_function()
def lf_vasc_medication(x):
    relevant_sections_medication = ["MEDICATIE", "ONTSLAG_MEDICATIE"]

    text = " ".join(
        str(x["sections"].get(sec, "")).lower() for sec in relevant_sections_medication
    )

    if any(re.search(p, text) for p in VASCULAR_MEDICATIONS):
        return PRESENT
    return ABSTAIN


lfs_for_component["V"] = [
    lf_vasc_negation,
    lf_vasc_positive,
    lf_vasc_medication,
    lf_vasc_family,
]

In [ ]:
# comp = 'V'
# lfs = lfs_for_component[comp]
# applier = PandasLFApplier(lfs=lfs)
# L = applier.apply(df)
# LFAnalysis(L, lfs_for_component["V"]).lf_summary()

In [ ]:
# df_copy = df.copy()
# df_copy['lf_vasc_family'] = df_copy.apply(lf_vasc_family, axis=1)

# family_triggered = df_copy[df_copy['lf_vasc_family'] == ABSENT]

# print(f"Family LF triggered in {len(family_triggered)} rows.\n")

# for i, row in family_triggered.head(10).iterrows():
#     text = ' '.join(str(row['sections'].get(sec, '')) for sec in relevant_sections)
#     print(f"--- Row {i} ---")
#     print(text)
#     print("-------------------\n")

In [ ]:
# df_copy = df.copy()
# df_copy['lf_vasc_negation_result'] = df_copy.apply(lf_vasc_negation, axis=1)

# negation_triggered = df_copy[df_copy['lf_vasc_negation_result'] == ABSENT]

# print(f"Negation LF triggered in {len(negation_triggered)} rows.\n")

# for i, row in negation_triggered.head(10).iterrows():
#     text = ' '.join(str(row['sections'].get(sec, '')) for sec in relevant_sections)
#     print(f"--- Row {i} ---")
#     print(text)
#     print("-------------------\n")

## Labelling functions for ages and sex

In [ ]:
relevant_sections = [
    "BESLUIT",
]


@labeling_function()
def lf_age_75_plus(x):
    text = " ".join(
        str(x["sections"].get(sec, "")).lower() for sec in relevant_sections
    )
    if re.search(r"\b(\d{2})-jarige\b", text):
        age = int(re.search(r"\b(\d{2})-jarige\b", text).group(1))
        if age >= 75:
            return PRESENT
        else:
            return ABSENT
    return ABSTAIN


@labeling_function()
def lf_age_65_74(x):
    text = " ".join(
        str(x["sections"].get(sec, "")).lower() for sec in relevant_sections
    )
    m = re.search(r"\b(\d{2})-jarige\b", text)
    if m:
        age = int(m.group(1))
        if 65 <= age < 75:
            return PRESENT
        else:
            return ABSENT
    return ABSTAIN


relevant_sections = [
    "ANAMNESE",
    "BESLUIT",
    "ANTECEDENTEN",
    "RISICOFACTOREN",
]


@labeling_function()
def lf_female_sex(x):
    text = " ".join(
        str(x["sections"].get(sec, "")).lower() for sec in relevant_sections
    )
    if re.search(r"\bpatiënte\b", text):
        return PRESENT
    elif re.search(r"\bpatiënt\b", text):
        return ABSENT
    return ABSTAIN


lfs_for_component["A2"] = [lf_age_75_plus]
lfs_for_component["A"] = [lf_age_65_74]
lfs_for_component["Sc"] = [lf_female_sex]

In [ ]:
# comp = 'Sc'
# lfs = lfs_for_component[comp]
# applier = PandasLFApplier(lfs=lfs)
# L = applier.apply(df)
# LFAnalysis(L, lfs_for_component[comp]).lf_summary()

# Creating the weakly labelled dataset with the LabelModel / MajorityVoter 
The following cells are dedicated for generating the weak training labels that can be used as input for a discriminative model. 
Switching between labelmodel and majorityvoter can be done by uncommenting the else clause of the wanted technique.

In [ ]:
val_df = pd.read_feather(r"..\..\Preprocessing\datasets\val_set.feather")
cols = ["CHF", "HTN", "AGE_75", "DM", "STROKE", "VASCULAR", "AGE_65_74", "SEX_FEMALE"]


percentages = {}
for col in cols:
    percentages[col] = (val_df[col] == 1).mean()

percentages = pd.Series(percentages).round(2)

print(percentages)

CHF           0.65
HTN           0.66
AGE_75        0.58
DM            0.26
STROKE        0.14
VASCULAR      0.51
AGE_65_74     0.24
SEX_FEMALE    0.42
dtype: float64


In [ ]:
components = ["C", "H", "A2", "D", "S", "V", "A", "Sc"]
PRIORS = dict(zip(components, percentages.values))
print(PRIORS)

{'C': np.float64(0.65), 'H': np.float64(0.66), 'A2': np.float64(0.58), 'D': np.float64(0.26), 'S': np.float64(0.14), 'V': np.float64(0.51), 'A': np.float64(0.24), 'Sc': np.float64(0.42)}


In [ ]:
soft_labels_dict = {}
components = ["C", "H", "A2", "D", "S", "V", "A", "Sc"]


for comp in components:
    lfs = lfs_for_component[comp]
    applier = PandasLFApplier(lfs=lfs)
    L = applier.apply(df)
    LFAnalysis(L, lfs).lf_summary()

    if len(lfs) == 1:
        soft_labels = L[:, 0].astype(float)
        # Replace ABSTAIN with prior chance of being PRESENT
        soft_labels[soft_labels == -1] = PRIORS[comp]

    # else:
    # label_model = LabelModel()
    # # label_model.fit(L_train=L, n_epochs=600, log_freq=100, seed=42)
    # probs = label_model.predict_proba(L)
    # soft_labels = probs[:,1].copy()

    else:
        label_model = MajorityLabelVoter()
        probs = label_model.predict_proba(L)
        soft_labels = probs[:, 1].copy()

        # Identify rows where EVERY LF abstained (all entries are -1)
        all_abstain_mask = (L == -1).all(axis=1)

        if comp in ["A2", "A", "Sc"]:
            # For demographics: Replace only "all abstains" with the prior
            soft_labels[all_abstain_mask] = PRIORS[comp]
        else:
            # For conditions: Replace only "all abstains" with 0
            soft_labels[all_abstain_mask] = 0

        # Note: True ties (e.g., one 0 and one 1) remain 0.5,
        # expressing the conflict accurately.

    soft_labels_dict[comp] = soft_labels

soft_label_array = np.stack([soft_labels_dict[comp] for comp in components], axis=1)
df["soft_label"] = list(soft_label_array)

100%|██████████| 4618/4618 [00:00<00:00, 7939.39it/s]


In [ ]:
soft_df = pd.DataFrame(df["soft_label"].tolist(), columns=components)
print(soft_df.head(40))

      C    H    A2    D    S    V     A   Sc
0   0.0  0.0  0.00  0.0  0.0  0.0  0.00  0.0
1   0.0  0.0  1.00  0.0  0.0  0.0  0.00  0.0
2   0.0  1.0  1.00  0.5  1.0  0.5  0.00  0.0
3   0.0  1.0  1.00  0.0  1.0  0.5  0.00  0.0
4   1.0  1.0  1.00  0.0  1.0  1.0  0.00  0.0
5   0.0  1.0  0.00  1.0  1.0  1.0  1.00  1.0
6   0.0  1.0  0.00  0.0  0.0  0.0  1.00  0.0
7   1.0  1.0  1.00  0.0  0.0  0.0  0.00  1.0
8   1.0  1.0  1.00  0.0  0.0  1.0  0.00  1.0
9   1.0  0.0  0.58  0.0  0.0  0.0  0.24  0.0
10  0.0  1.0  0.00  1.0  0.0  1.0  1.00  0.0
11  0.0  0.0  0.00  0.0  0.0  0.0  0.00  0.0
12  0.0  0.0  0.00  0.0  1.0  1.0  0.00  0.0
13  1.0  0.0  1.00  0.0  0.0  1.0  0.00  1.0
14  0.0  1.0  0.00  0.0  0.0  1.0  1.00  0.0
15  0.0  1.0  1.00  0.0  0.0  0.0  0.00  0.0
16  1.0  1.0  0.00  1.0  1.0  0.0  0.00  0.0
17  1.0  1.0  0.00  1.0  0.0  1.0  1.00  1.0
18  0.0  0.0  0.00  0.0  1.0  1.0  0.00  0.0
19  0.0  1.0  0.00  0.0  0.0  0.0  0.00  1.0
20  0.0  1.0  0.00  1.0  0.0  0.0  1.00  0.0
21  1.0  0

In [ ]:
soft_df.to_feather(r"trainingsets/model.feather")

# Labelling functions directly on the test set / train set
We apply the labeling functions directly on the test set, so we can see what the impact is without BERT trained on it.  
We use the priors calculated from the previous section, based on the validation set. 

In [ ]:
TESTFILE = r"..\..\Preprocessing\datasets\test_set.feather"
test_df = pd.read_feather(TESTFILE)
val_df = pd.read_feather(r"..\..\Preprocessing\datasets\val_set.feather")

components = ["C", "H", "A2", "D", "S", "V", "A", "Sc"]

soft_labels_dict = {}

for comp in components:
    lfs = lfs_for_component[comp]
    applier = PandasLFApplier(lfs=lfs)
    L = applier.apply(val_df)
    LFAnalysis(L, lfs).lf_summary()

    if len(lfs) == 1:
        soft_labels = L[:, 0].astype(float)
        soft_labels[soft_labels == -1] = PRIORS[comp]
    # else:
    #     label_model = LabelModel()
    #     label_model.fit(L_train=L, n_epochs=500, log_freq=100, seed=42)
    #     probs = label_model.predict_proba(L)
    #     soft_labels = probs[:,1]

    else:
        label_model = MajorityLabelVoter()
        probs = label_model.predict_proba(L)
        soft_labels = probs[:, 1].copy()

        # Identify rows where EVERY LF abstained (all entries are -1)
        all_abstain_mask = (L == -1).all(axis=1)

        if comp in ["A2", "A", "Sc"]:
            # For demographics: Replace only "all abstains" with the prior
            soft_labels[all_abstain_mask] = PRIORS[comp]
        else:
            # For conditions: Replace only "all abstains" with 0
            soft_labels[all_abstain_mask] = 0

        # Note: True ties (e.g., one 0 and one 1) remain 0.5,
        # expressing the conflict accurately.

    soft_labels_dict[comp] = soft_labels

soft_label_array = np.stack([soft_labels_dict[comp] for comp in components], axis=1)
val_df["soft_label"] = list(soft_label_array)


soft_labels_dict = {}

for comp in components:
    lfs = lfs_for_component[comp]
    applier = PandasLFApplier(lfs=lfs)
    L = applier.apply(test_df)
    LFAnalysis(L, lfs).lf_summary()

    lf_names = [lf.name for lf in lfs]

    def get_fired_lfs(row_index):
        # Get indices where the LF did not abstain (abstain = -1)
        active_indices = np.where(L[row_index, :] != -1)[0]
        # Return a list of the names of those LFs
        return [lf_names[i] for i in active_indices]

    # Store the list of firing LFs in a new column
    test_df[f"fired_lfs_{comp}"] = [get_fired_lfs(i) for i in range(len(test_df))]

    if len(lfs) == 1:
        soft_labels = L[:, 0].astype(float)
        soft_labels[soft_labels == -1] = PRIORS[comp]
    # else:
    #     label_model = LabelModel()
    #     label_model.fit(L_train=L, n_epochs=500, log_freq=100, seed=42)
    #     probs = label_model.predict_proba(L)
    #     soft_labels = probs[:,1]

    else:
        label_model = MajorityLabelVoter()
        probs = label_model.predict_proba(L)
        soft_labels = probs[:, 1].copy()

        # Identify rows where EVERY LF abstained (all entries are -1)
        all_abstain_mask = (L == -1).all(axis=1)

        if comp in ["A2", "A", "Sc"]:
            # For demographics: Replace only "all abstains" with the prior
            soft_labels[all_abstain_mask] = PRIORS[comp]
        else:
            # For conditions: Replace only "all abstains" with 0
            soft_labels[all_abstain_mask] = 0

        # Note: True ties (e.g., one 0 and one 1) remain 0.5,
        # expressing the conflict accurately.

    soft_labels_dict[comp] = soft_labels

soft_label_array = np.stack([soft_labels_dict[comp] for comp in components], axis=1)
test_df["soft_label"] = list(soft_label_array)

100%|██████████| 200/200 [00:00<00:00, 6135.07it/s]


In [ ]:
soft_df = pd.DataFrame(test_df["soft_label"].tolist(), columns=components)
print(soft_df.head(40))

      C    H    A2    D    S    V     A   Sc
0   1.0  1.0  1.00  0.0  0.0  1.0  0.00  1.0
1   0.0  0.0  0.00  0.0  0.0  0.0  0.00  1.0
2   0.0  1.0  0.00  0.0  0.0  1.0  0.00  0.0
3   0.0  1.0  1.00  1.0  0.0  0.0  0.00  1.0
4   1.0  1.0  1.00  0.0  0.0  0.0  0.00  0.0
5   0.0  0.0  0.00  0.0  1.0  0.0  1.00  0.0
6   1.0  1.0  0.00  1.0  0.0  1.0  1.00  1.0
7   1.0  0.0  1.00  0.0  0.0  1.0  0.00  0.0
8   0.0  0.0  0.00  0.0  0.0  1.0  1.00  0.0
9   0.0  0.0  0.00  0.0  0.0  0.0  0.00  1.0
10  0.0  1.0  0.00  0.0  0.0  0.0  0.00  0.0
11  0.0  1.0  0.00  0.0  0.0  0.0  1.00  0.0
12  1.0  1.0  0.00  1.0  0.0  1.0  1.00  0.0
13  1.0  1.0  1.00  1.0  0.0  1.0  0.00  0.0
14  1.0  1.0  0.00  0.0  0.0  0.0  0.00  0.0
15  1.0  1.0  1.00  0.0  0.0  0.0  0.00  0.0
16  0.0  1.0  1.00  1.0  0.0  0.0  0.00  1.0
17  1.0  0.0  1.00  0.0  1.0  0.0  0.00  1.0
18  0.5  0.0  1.00  0.0  0.0  0.0  0.00  0.0
19  1.0  1.0  0.00  1.0  0.0  0.0  1.00  0.0
20  1.0  1.0  1.00  0.0  1.0  1.0  0.00  1.0
21  1.0  1

In [ ]:
print("Number and percentage of soft labels assigned by prior probability:\n")

for comp in components:
    soft_labels = soft_labels_dict[comp]
    prior_value = PRIORS[comp]

    n_prior = np.sum(soft_labels == prior_value)
    total = len(soft_labels)
    pct_prior = n_prior / total * 100

    print(f"{comp}: {n_prior}/{total} → {pct_prior:.1f}% assigned prior")

Number and percentage of soft labels assigned by prior probability:

C: 0/200 → 0.0% assigned prior
H: 0/200 → 0.0% assigned prior
A2: 14/200 → 7.0% assigned prior
D: 0/200 → 0.0% assigned prior
S: 0/200 → 0.0% assigned prior
V: 0/200 → 0.0% assigned prior
A: 14/200 → 7.0% assigned prior
Sc: 1/200 → 0.5% assigned prior


Calculate thresholds based on manually labelled validation set. 

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

GT_COLS = {
    "C": "CHF",
    "H": "HTN",
    "A2": "AGE_75",
    "D": "DM",
    "S": "STROKE",
    "V": "VASCULAR",
    "A": "AGE_65_74",
    "Sc": "SEX_FEMALE",
}

best_thresholds = {}

soft_val_df = pd.DataFrame(val_df["soft_label"].tolist(), columns=components)
cols = list(GT_COLS.values())

val_df[cols] = val_df[cols].fillna(0).astype(int)


for comp in GT_COLS.keys():
    y_true = val_df[GT_COLS[comp]]
    y_probs = soft_val_df[comp]

    thresholds = np.linspace(0.01, 0.99, 99)
    best_f1 = 0
    best_t = 0.5

    for t in thresholds:
        y_pred = (y_probs >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    best_thresholds[comp] = best_t

print(best_thresholds)

{'C': np.float64(0.51), 'H': np.float64(0.51), 'A2': np.float64(0.5800000000000001), 'D': np.float64(0.51), 'S': np.float64(0.01), 'V': np.float64(0.01), 'A': np.float64(0.24000000000000002), 'Sc': np.float64(0.42000000000000004)}


In [ ]:
## This cell can only be ran when evaluating the test set, since hard labels are required.

from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    accuracy_score,
    classification_report,
)
import numpy as np
import pandas as pd

GT_COLS = {
    "C": "CHF",
    "H": "HTN",
    "A2": "AGE_75",
    "D": "DM",
    "S": "STROKE",
    "V": "VASCULAR",
    "A": "AGE_65_74",
    "Sc": "SEX_FEMALE",
}

label_cols = list(GT_COLS.values())


test_preds = []
test_lbls = []
per_component_results = []

for comp, gt_col in GT_COLS.items():

    y_probs = soft_df[comp]
    # y_probs = test_df['soft_label'].apply(lambda x: x[i])

    y_pred = (y_probs >= best_thresholds[comp]).astype(int)
    y_true = test_df[gt_col].astype(int)

    test_preds.append(y_pred.values)
    test_lbls.append(y_true.values)

    # Per-label metrics
    per_component_results.append(
        {
            "Component": comp,
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall": recall_score(y_true, y_pred, zero_division=0),
            "F1": f1_score(y_true, y_pred, zero_division=0),
        }
    )

# Convert to proper multilabel shape (n_samples, n_labels)
test_preds = np.vstack(test_preds).T
test_lbls = np.vstack(test_lbls).T


metrics = {
    "Micro-F1": f1_score(test_lbls, test_preds, average="micro"),
    "Macro-F1": f1_score(test_lbls, test_preds, average="macro"),
    "Micro-Precision": precision_score(test_lbls, test_preds, average="micro"),
    "Micro-Recall": recall_score(test_lbls, test_preds, average="micro"),
    "Macro-Precision": precision_score(test_lbls, test_preds, average="macro"),
    "Macro-Recall": recall_score(test_lbls, test_preds, average="macro"),
    "Hamming Loss": hamming_loss(test_lbls, test_preds),
    "Subset Accuracy": accuracy_score(test_lbls, test_preds),
}

print("\n===== GLOBAL MULTILABEL METRICS =====")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

print("\n===== PER-LABEL REPORT =====")
print(
    classification_report(
        test_lbls, test_preds, target_names=GT_COLS.keys(), zero_division=0, digits=3
    )
)


lm_results_df = pd.DataFrame(per_component_results)

print("\n===== PER-COMPONENT SUMMARY =====")
print(lm_results_df.round(3))


===== GLOBAL MULTILABEL METRICS =====
Micro-F1: 0.9538
Macro-F1: 0.9430
Micro-Precision: 0.9289
Micro-Recall: 0.9800
Macro-Precision: 0.9129
Macro-Recall: 0.9803
Hamming Loss: 0.0356
Subset Accuracy: 0.7650

===== PER-LABEL REPORT =====
              precision    recall  f1-score   support

           C      0.916     0.990     0.951        99
           H      1.000     0.993     0.996       135
          A2      1.000     0.990     0.995        96
           D      0.767     1.000     0.868        46
           S      0.821     1.000     0.901        32
           V      0.851     0.955     0.900        66
           A      1.000     0.979     0.989        47
          Sc      0.949     0.937     0.943        79

   micro avg      0.929     0.980     0.954       600
   macro avg      0.913     0.980     0.943       600
weighted avg      0.936     0.980     0.956       600
 samples avg      0.888     0.928     0.901       600


===== PER-COMPONENT SUMMARY =====
  Component  Precision

In [ ]:
final_decisions = pd.DataFrame(index=test_df.index)
final_decisions["text"] = test_df["text"]
final_decisions["unique_id"] = test_df["unique_id"]
fired_cols = [col for col in test_df.columns if col.startswith("fired_lfs_")]
final_decisions[fired_cols] = test_df[fired_cols]

for i, comp in enumerate(components):
    y_probs = test_df["soft_label"].apply(lambda x: x[i])

    y_pred = (y_probs >= best_thresholds[comp]).astype(int)

    final_decisions[f"pred_{comp}"] = y_pred

# Verification
print(f"Final decisions shape: {final_decisions.shape}")
print(f"Any NaNs? {final_decisions.isna().sum().sum()}")


final_decisions.to_feather("results/snorkel_directly_on_train_mv.feather")

Final decisions shape: (200, 18)
Any NaNs? 0
